In [141]:
import pandas as pd
from pathlib import Path
import sys
import os

notebook_dir = Path(os.getcwd())
parent_dir = str(notebook_dir.parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import scripts.data_pull_functions 
import scripts.gather_historic_data 
import scripts.gather_data_to_forecast 
import scripts.model_training_functions 
import scripts.create_output_functions 

import importlib

importlib.reload(scripts.data_pull_functions)
importlib.reload(scripts.gather_historic_data)
importlib.reload(scripts.gather_data_to_forecast)
importlib.reload(scripts.model_training_functions)
importlib.reload(scripts.create_output_functions)
importlib.reload(scripts.constants)

from scripts.data_pull_functions import login_google_cloud
from scripts.gather_historic_data import gather_historic_data, pull_historic_unit_availability
from scripts.gather_data_to_forecast import gather_data_to_forecast
from scripts.model_training_functions import fit_xgboost_clarissa_version, generate_feed_forward_forecast
from scripts.create_output_functions import create_nbpl_file, create_next_day_gas_burn_file
from scripts.constants import SITES_LIST, SITES_OR_LIST, AVAILABILITY_FILES_XLSX, AVAILABILITY_FILES_CSV

In [138]:

#SITES = ["CGS", "DCS", "LCS", "PGS", "GGS"]
SITES_OR_LIST = '|'.join(["CGS", "DCS", "LCS", "PGS", "GGS"])
SITES_OR_LIST

'CGS|DCS|LCS|PGS|GGS'

In [5]:
folder_location = Path("G:\\Trading\\Forecasts\\Daily Gas Burn Forecast by Site\\Unit Availability Exports - Future")

In [6]:
latest_file = Path(max(folder_location.glob('*'), key=lambda f: f.stat().st_birthtime))
data = pd.read_csv(latest_file).iloc[:, 1:]
print(data.shape)
data.head()

(197, 241)


,Name,HE1,HE2,HE3,HE4,HE5,HE6,HE7,HE8,HE9,...,HE15,HE16,HE17,HE18,HE19,HE20,HE21,HE22,HE23,HE24
0,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Baseload Generation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LRS1_X.LRSE - Econ High,560.0,560.0,560.0,560.0,560.0,560.0,560.0,560.0,560.0,...,560.0,560.0,560.0,560.0,560.0,560.0,560.0,560.0,560.0,560.0
3,LRS1_X.LRSE - High Effective Limit,560.0,560.0,560.0,560.0,560.0,560.0,560.0,560.0,560.0,...,560.0,560.0,560.0,560.0,560.0,560.0,560.0,560.0,560.0,560.0
4,AVS1 - Econ High,435.0,435.0,435.0,435.0,435.0,435.0,435.0,435.0,435.0,...,435.0,435.0,435.0,435.0,435.0,435.0,435.0,435.0,435.0,435.0


In [175]:
transposed = data[ (data['Name'].str.contains(SITES_OR_LIST, na=False)) & (data['Name'].str.contains('High Effective Limit', na=False))].transpose()
col_names = transposed.iloc[0,:]


In [176]:
transposed.head()

,19,21,23,25,27,29,31,33,35,37,...,65,67,69,71,73,75,77,79,81,83
Name,CGS1 - High Effective Limit,DCS1 - High Effective Limit,GGS1 - High Effective Limit,GGS2 - High Effective Limit,LCS1 - High Effective Limit,LCS2 - High Effective Limit,LCS3 - High Effective Limit,LCS4 - High Effective Limit,LCS5 - High Effective Limit,LCS6 - High Effective Limit,...,PGS21 - High Effective Limit,PGS22 - High Effective Limit,PGS31 - High Effective Limit,PGS32 - High Effective Limit,PGS33 - High Effective Limit,PGS34 - High Effective Limit,PGS35 - High Effective Limit,PGS36 - High Effective Limit,PGS4 - High Effective Limit,PGS5 - High Effective Limit
HE1,0.0,295.0,68.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,218.62,0.0
HE2,0.0,295.0,68.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,214.52,0.0
HE3,0.0,295.0,68.0,58.0,35.0,0.0,0.0,42.0,42.0,42.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,215.4,0.0
HE4,0.0,295.0,92.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,216.12,0.0


In [ ]:
unit_availability_df = transposed.iloc[1:,:]
unit_availability_df.columns = col_names
unit_availability_df.head()

Name,CGS1 - High Effective Limit,DCS1 - High Effective Limit,GGS1 - High Effective Limit,GGS2 - High Effective Limit,LCS1 - High Effective Limit,LCS2 - High Effective Limit,LCS3 - High Effective Limit,LCS4 - High Effective Limit,LCS5 - High Effective Limit,LCS6 - High Effective Limit,...,PGS21 - High Effective Limit,PGS22 - High Effective Limit,PGS31 - High Effective Limit,PGS32 - High Effective Limit,PGS33 - High Effective Limit,PGS34 - High Effective Limit,PGS35 - High Effective Limit,PGS36 - High Effective Limit,PGS4 - High Effective Limit,PGS5 - High Effective Limit
HE1,0.0,295.0,68.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,218.62,0.0
HE2,0.0,295.0,68.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,214.52,0.0
HE3,0.0,295.0,68.0,58.0,35.0,0.0,0.0,42.0,42.0,42.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,215.4,0.0
HE4,0.0,295.0,92.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,216.12,0.0
HE5,0.0,296.0,93.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,217.0,0.0


In [178]:
words = str(latest_file).removesuffix(".csv").split("_")
start_time = words[-2]
start_time

'2026090200'

In [179]:
end_time = words[-1]
end_time

'2026091023'

In [180]:
start_year = start_time[:4]
start_month = start_time[4:6]
start_day = start_time[6:8]
start_hour = start_time[8:10]

end_year = end_time[:4]
end_month = end_time[4:6]
end_day = end_time[6:8]
end_hour = end_time[8:10]

print(f'{start_year}  {start_month}  {start_day}  {start_hour}')
print(f'{end_year}  {end_month}  {end_day}  {end_hour}')

2026  09  02  00
2026  09  10  23


In [181]:
datetime_start = pd.Timestamp(year=int(start_year), month=int(start_month), day=int(start_day), hour=int(start_hour))
datetime_end = pd.Timestamp(year=int(end_year), month=int(end_month), day=int(end_day), hour=int(end_hour))

In [182]:
datetime_start

Timestamp('2026-09-02 00:00:00')

In [183]:
datetime_end

Timestamp('2026-09-10 23:00:00')

In [184]:
datetimes = pd.date_range(start=datetime_start, end=datetime_end, freq='h')

In [185]:
transposed

Name,CGS1 - High Effective Limit,DCS1 - High Effective Limit,GGS1 - High Effective Limit,GGS2 - High Effective Limit,LCS1 - High Effective Limit,LCS2 - High Effective Limit,LCS3 - High Effective Limit,LCS4 - High Effective Limit,LCS5 - High Effective Limit,LCS6 - High Effective Limit,...,PGS21 - High Effective Limit,PGS22 - High Effective Limit,PGS31 - High Effective Limit,PGS32 - High Effective Limit,PGS33 - High Effective Limit,PGS34 - High Effective Limit,PGS35 - High Effective Limit,PGS36 - High Effective Limit,PGS4 - High Effective Limit,PGS5 - High Effective Limit
HE1,0.0,295.0,68.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,218.62,0.0
HE2,0.0,295.0,68.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,214.52,0.0
HE3,0.0,295.0,68.0,58.0,35.0,0.0,0.0,42.0,42.0,42.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,215.4,0.0
HE4,0.0,295.0,92.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,216.12,0.0
HE5,0.0,296.0,93.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,217.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HE20,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0
HE21,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0
HE22,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0
HE23,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0


In [186]:
transposed = transposed.reset_index(drop=True)
transposed

Name,CGS1 - High Effective Limit,DCS1 - High Effective Limit,GGS1 - High Effective Limit,GGS2 - High Effective Limit,LCS1 - High Effective Limit,LCS2 - High Effective Limit,LCS3 - High Effective Limit,LCS4 - High Effective Limit,LCS5 - High Effective Limit,LCS6 - High Effective Limit,...,PGS21 - High Effective Limit,PGS22 - High Effective Limit,PGS31 - High Effective Limit,PGS32 - High Effective Limit,PGS33 - High Effective Limit,PGS34 - High Effective Limit,PGS35 - High Effective Limit,PGS36 - High Effective Limit,PGS4 - High Effective Limit,PGS5 - High Effective Limit
0,0.0,295.0,68.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,218.62,0.0
1,0.0,295.0,68.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,214.52,0.0
2,0.0,295.0,68.0,58.0,35.0,0.0,0.0,42.0,42.0,42.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,215.4,0.0
3,0.0,295.0,92.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,216.12,0.0
4,0.0,296.0,93.0,58.0,35.0,0.0,0.0,41.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,217.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0
212,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0
213,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0
214,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0


In [188]:
transposed.insert(loc=0, column='datetime', value=datetimes)

In [190]:
transposed

Name,datetime,CGS1 - High Effective Limit,DCS1 - High Effective Limit,GGS1 - High Effective Limit,GGS2 - High Effective Limit,LCS1 - High Effective Limit,LCS2 - High Effective Limit,LCS3 - High Effective Limit,LCS4 - High Effective Limit,LCS5 - High Effective Limit,...,PGS21 - High Effective Limit,PGS22 - High Effective Limit,PGS31 - High Effective Limit,PGS32 - High Effective Limit,PGS33 - High Effective Limit,PGS34 - High Effective Limit,PGS35 - High Effective Limit,PGS36 - High Effective Limit,PGS4 - High Effective Limit,PGS5 - High Effective Limit
0,2026-09-02 00:00:00,0.0,295.0,68.0,58.0,35.0,0.0,0.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,218.62,0.0
1,2026-09-02 01:00:00,0.0,295.0,68.0,58.0,35.0,0.0,0.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,214.52,0.0
2,2026-09-02 02:00:00,0.0,295.0,68.0,58.0,35.0,0.0,0.0,42.0,42.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,215.4,0.0
3,2026-09-02 03:00:00,0.0,295.0,92.0,58.0,35.0,0.0,0.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,216.12,0.0
4,2026-09-02 04:00:00,0.0,296.0,93.0,58.0,35.0,0.0,0.0,41.0,41.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,217.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,2026-09-10 19:00:00,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0
212,2026-09-10 20:00:00,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0
213,2026-09-10 21:00:00,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0
214,2026-09-10 22:00:00,0.0,297.0,95.0,95.0,35.0,45.0,45.0,45.0,45.0,...,8.9,8.9,18.6,18.6,18.6,18.6,18.6,18.6,225.0,0.0


In [149]:
object_cols = transposed.select_dtypes(include=['object']).columns
object_cols

Index(['    CGS1 - High Effective Limit', '    DCS1 - High Effective Limit',
       '    GGS1 - High Effective Limit', '    GGS2 - High Effective Limit',
       '    LCS1 - High Effective Limit', '    LCS2 - High Effective Limit',
       '    LCS3 - High Effective Limit', '    LCS4 - High Effective Limit',
       '    LCS5 - High Effective Limit', '    LCS6 - High Effective Limit',
       '    PGS1 - High Effective Limit', '    PGS2 - High Effective Limit',
       '    PGS3 - High Effective Limit', '    PGS11 - High Effective Limit',
       '    PGS12 - High Effective Limit', '    PGS13 - High Effective Limit',
       '    PGS14 - High Effective Limit', '    PGS15 - High Effective Limit',
       '    PGS16 - High Effective Limit', '    PGS17 - High Effective Limit',
       '    PGS18 - High Effective Limit', '    PGS19 - High Effective Limit',
       '    PGS20 - High Effective Limit', '    PGS21 - High Effective Limit',
       '    PGS22 - High Effective Limit', '    PGS31 - High Effe

In [150]:
transposed[object_cols] = transposed[object_cols].apply(pd.to_numeric, errors='coerce')

In [151]:
transposed.info()

<class 'pandas.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 33 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0       CGS1 - High Effective Limit   216 non-null    float64
 1       DCS1 - High Effective Limit   216 non-null    float64
 2       GGS1 - High Effective Limit   216 non-null    float64
 3       GGS2 - High Effective Limit   216 non-null    float64
 4       LCS1 - High Effective Limit   216 non-null    float64
 5       LCS2 - High Effective Limit   216 non-null    float64
 6       LCS3 - High Effective Limit   216 non-null    float64
 7       LCS4 - High Effective Limit   216 non-null    float64
 8       LCS5 - High Effective Limit   216 non-null    float64
 9       LCS6 - High Effective Limit   216 non-null    float64
 10      PGS1 - High Effective Limit   216 non-null    float64
 11      PGS2 - High Effective Limit   216 non-null    float64
 12      PGS3 - High Eff

In [129]:
def _transpose_raw_unit_availability(raw_df: pd.DataFrame = None,
                                    datetimes: pd.DatetimeIndex = None,
                                    start_date = None,
                                    end_date = None):
    
    
    unit_availability_df = raw_df[ (raw_df['Name'].str.contains(SITES_OR_LIST, na=False)) & (raw_df['Name'].str.contains('High Effective Limit', na=False))].transpose()
    col_names = unit_availability_df.iloc[0,:]
    unit_availability_df = unit_availability_df.iloc[1:,:]
    unit_availability_df.columns = col_names
    unit_availability_df = unit_availability_df.reset_index(drop=True)
    unit_availability_df.insert(loc=0, column='datetime', value=datetimes)
    object_cols = unit_availability_df.select_dtypes(include=['object']).columns
    unit_availability_df[object_cols] = unit_availability_df[object_cols].apply(pd.to_numeric, errors='coerce')

    #replicating data cleaning for historic data
    #Sorting chronologically
    unit_availability_df = (unit_availability_df.sort_values("datetime").reset_index(drop=True))
    
    #Data cleaning and formatting
    unit_availability_df.columns = unit_availability_df.columns.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

    unit_availability_df = unit_availability_df.dropna(subset = ['datetime'], axis=0) # drops rows that don't have dates due to daylight savings shifts

    value_cols = [col for col in unit_availability_df.columns if isinstance(col, str) and "High Effective Limit" in col]

    unit_availability_df = unit_availability_df.melt(id_vars=["datetime"], value_vars=value_cols, var_name="unit", value_name="availability_mw")
    
    unit_availability_df["site"] = unit_availability_df["unit"].str.upper().str.extract(r"^(CGS|DCS|GGS|LCS|PGS)", expand=False).str.strip().str.upper()
    unit_availability_df = unit_availability_df.sort_values(by=["site", "unit", "datetime"])[["datetime","site", "unit", "availability_mw"]]
    unit_availability_df = unit_availability_df[(unit_availability_df['datetime'] >= start_date) & (unit_availability_df['datetime'] <= end_date)]

    
    site_availability_df = unit_availability_df.groupby(["datetime", 'site']).agg({"availability_mw": "sum"}).reset_index()
    site_availability_df = site_availability_df.sort_values(by=["site", "datetime"])[["datetime","site", "availability_mw"]]
    site_availability_df = site_availability_df[(site_availability_df['datetime'] >= start_date) & (site_availability_df['datetime'] <= end_date)]

    return {
        "unit_availability_df": unit_availability_df,
        "site_availability_df": site_availability_df
        }

In [130]:
def pull_and_transpose_raw_unit_availability(folder_location = 'G:\\Trading\\Forecasts\\Daily Gas Burn Forecast by Site\\Unit Availability Exports - Future',
                                             start_date = '2026-09-04',
                                             end_date = '2026-09-13'):


    folder_location = Path(folder_location)
    most_recent_file = Path(max(folder_location.glob('*'), key=lambda f: f.stat().st_birthtime))

    #getting start and end dates from file name
    file_name_words = str(most_recent_file).removesuffix(".csv").split("_")
    start_time = file_name_words[-2]
    end_time = file_name_words[-1]

    # file name contains start and end times in YYYYMMDDHH format. Extracting that info and creating a datetime range
    datetime_start = pd.Timestamp(year=int(start_time[:4]), month=int(start_time[4:6]), day=int(start_time[6:8]), hour=int(start_time[8:10]))
    datetime_end = pd.Timestamp(year=int(end_time[:4]), month=int(end_time[4:6]), day=int(end_time[6:8]), hour=int(end_time[8:10]))
    datetimes = pd.date_range(start=datetime_start, end=datetime_end, freq='h')

    raw_df = pd.read_csv(most_recent_file).iloc[:, 1:]
    
    transposed_dfs = _transpose_raw_unit_availability(raw_df=raw_df, start_date=start_date, end_date=end_date, datetimes=datetimes)

    return transposed_dfs

In [134]:
data = pull_and_transpose_raw_unit_availability()
data = data['site_availability_df']


In [135]:
data.tail()

,datetime,site,availability_mw
1059,2026-09-12 19:00:00,PGS,785.6
1064,2026-09-12 20:00:00,PGS,785.6
1069,2026-09-12 21:00:00,PGS,785.6
1074,2026-09-12 22:00:00,PGS,785.6
1079,2026-09-12 23:00:00,PGS,785.6


In [136]:
data['availability_mw'].sum()

np.float64(275282.79000000004)

In [137]:
old_version = pd.read_csv("C:\\Users\\A105158\\OneDrive - Basin Electric Power Cooperative\\Desktop\\Python Projects\\gas-burn-by-site\\data\\processed-data\\data_to_forecast_df.csv")
old_version['availability_mw'].sum()

np.float64(275282.79000000004)